explore the cleaned data and the preprocessing phase

In [1]:
from src.preprocess import remove_offsets

In [2]:
imu, offsets = remove_offsets()
offsets

file              column
imu_acceleration  x        -0.072248
imu_angular_rate  x         1.076229
                  y        -0.276122
                  z         1.011456
dtype: float64

Since the method we used was to take the average of means of offset during each window so the offset to substract matches our values we set in issues.csv

In [4]:
imu["imu_angular_rate"]

,t_s,_timestamp,x,y,z
0,0.001332,1725799119377875,0.113771,0.076122,0.038544
1,0.005799,1725799119382342,0.133771,0.166122,0.048544
2,0.010266,1725799119386809,0.123771,0.156122,0.018544
3,0.014807,1725799119391350,0.223771,-0.013878,0.078544
4,0.019228,1725799119395771,-0.096229,0.196122,0.078544
...,...,...,...,...,...
489824,2206.949995,1725801326326538,-0.166229,0.026122,-0.041456
489825,2206.954010,1725801326330553,0.073771,-0.073878,0.018544
489826,2206.958487,1725801326335030,-0.376229,-0.043878,-0.011456
489827,2206.962945,1725801326339488,-0.246229,0.116122,-0.041456


In [5]:
imu["imu_acceleration"]

,t_s,_timestamp,x,y,z
0,0.002541,1725799119379084,0.002248,-0.02,-0.97
1,0.007002,1725799119383545,0.002248,-0.02,-0.97
2,0.011476,1725799119388019,0.002248,-0.02,-0.97
3,0.015953,1725799119392496,0.002248,-0.03,-0.97
4,0.020418,1725799119396961,0.002248,-0.03,-0.97
...,...,...,...,...,...
489824,2206.950817,1725801326327360,0.012248,-0.02,-0.99
489825,2206.955222,1725801326331765,0.012248,-0.01,-0.97
489826,2206.962301,1725801326338844,-0.007752,-0.02,-0.95
489827,2206.964155,1725801326340698,0.002248,-0.02,-0.98


imu_acc z is still ~-0.97 at rest

verify the unwrapping removed every wrap jump and nothing else.
* s: at line crossing the s jumps from ~805 to ~0, unwrapped s should continue smoothly past 805.
* during the pit stop, raw s flickers between ~805 and ~0 while the car stands need the line, unwrapped s shhould stay flat. this confirms the flicker doesnt create fake jumps.
- if both hold then the linear interpolation later will not produce values from the wrong side of the track

In [2]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from src.loader import load, load_centerline
from src.preprocess import unwrap

L = load_centerline().attrs["length"]
cc = load("vehicle_curvilinear_coordinates")
cc_u = unwrap(cc, "s", L)

t_cross = 348.448507   # lap start_s from find_laps, this is lap 4
windows = [
    (t_cross - 5, t_cross + 5, "Line crossing"),
    (970, 1200, "Pit stop (flicker)"),
]

fig = make_subplots(rows=2, cols=1, subplot_titles=[w[2] for w in windows])
for i, (t0, t1, _) in enumerate(windows, start=1):
    m = (cc["t_s"] >= t0) & (cc["t_s"] <= t1)
    fig.add_trace(go.Scatter(x=cc.loc[m, "t_s"], y=cc.loc[m, "s"],
                             name="raw s", line_color="gray",
                             showlegend=(i == 1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=cc_u.loc[m, "t_s"], y=cc_u.loc[m, "s"],
                             name="unwrapped s", line_color="crimson",
                             showlegend=(i == 1)), row=i, col=1)
    fig.update_yaxes(title_text="s (m)", row=i, col=1)

fig.update_xaxes(title_text="t_s (s)", row=2, col=1)
fig.update_layout(title="s before and after unwrap", height=700)
fig.show()

analysis:
* in the top graph: raw s climbs to abt 805m then drops back to 0 at the line and starts climbing again. whereas the red (unwrapped) s keeps climbing smoothly with no step. it sits at around 3200 bc it's the data from lap 4 so 4x805 at that point 
* in the bottom one, the pit stop, there is a flicker in the raw data when the car s sitting on the cross line which does not happen in the unwrapped (red) line and it stays flat at around 11200m which makes teh flicker disappear. 
- -> unwrap removed thereal wraps and the pit flicker. 

In [12]:
from src.preprocess import gap_intervals
from src.loader import load
FILES = [
    "hv_power",
    "imu_angular_rate",
    "imu_acceleration",
    "vehicle_position",
    "vehicle_speed",
    "vehicle_curvilinear_coordinates",
    "pedal_brakes_pressure",
    "pedal_throttle",
    "steer_angle",
    "front_angular_velocity",
    "inv_r_id_a8_n_actual_filt",
    "inv_l_id_a8_n_actual_filt",
    "inv_r_id_27_iq_actual",
    "inv_l_id_27_iq_actual",
]

dfs = {f: load(f) for f in FILES}

import pandas as pd
from src.config import OUT_LOGS   

profile = pd.read_csv(OUT_LOGS / "profile_files.csv").set_index("file")
intervals = gap_intervals(dfs)

for file, intv in intervals.items():
    expected = profile.loc[file, "n_gaps"]
    print(f"{file:35s} {len(intv):5d}  profile: {expected:5d}  {'OK' if len(intv) == expected else 'MISMATCH'}")

hv_power                             1289  profile:  1289  OK
imu_angular_rate                      277  profile:   277  OK
imu_acceleration                      285  profile:   285  OK
vehicle_position                       72  profile:    72  OK
vehicle_speed                          72  profile:    72  OK
vehicle_curvilinear_coordinates        72  profile:    72  OK
pedal_brakes_pressure                 487  profile:   487  OK
pedal_throttle                        956  profile:   956  OK
steer_angle                           653  profile:   653  OK
front_angular_velocity                128  profile:   128  OK
inv_r_id_a8_n_actual_filt              72  profile:    72  OK
inv_l_id_a8_n_actual_filt              72  profile:    72  OK
inv_r_id_27_iq_actual                  71  profile:    71  OK
inv_l_id_27_iq_actual                  69  profile:    69  OK


The number of gaps matches with the profiler so we're all good.

next step is to sampling, this is my thought process of how i came up to choosing the sampleing rate:
first, I think in order to avoid losing any data, it would be better to upsample than downsample.
the highest rate in our data is the imu with ~223Hz
the rate found from the profile was ~223, whereas in data.md it said the rate is ~235, from these two values the assumption is that data from another session the imu was better and could run faster. 
i.e. if another set of data would be introduced or if we were to use data from another session, the rate might be higher than 223 and hence instead of upsampling, we would be downampling without any filter. 
I would prefer for the code to work beyond this small dataset, so i'd rather choose a higher rate that would not cause any problems. 

a rate of 223.41 (current one) gives a timestep of 0.004476s, it would be *reasonable* to choose a 0.004s time step to be precise, which is a rate of 250Hz. this way neither this data or data from another session would result in change in the code.